# Scanpy Test

## Switch to Conda and Install Packages
#### These need to be run in Terminal
Switch kernel to conda (base python 3.12.13)

In [ ]:
# if necessary then run in terminal
conda activate base
pip install pandas numpy matplotlib seaborn
pip install 'scanpy[leiden]'
pip install anndata
pip install pooch

## Import packages into environment

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Core scverse libraries
import anndata as ad
from __future__ import annotations

# Data retrieval
import pooch as po
import scanpy as sc
sc.settings.set_figure_params(dpi=50, facecolor="white")

## Load Data and perform Pre-processing

In [ ]:
adata = sc.read_10x_h5("E:/Zhang Lab/Archives/RNASeq/RNASeq/GSE181902/EY003_cellranger_count_outs/filtered_feature_bc_matrix.h5")
adata.var_names_make_unique()
print(adata)

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.scrublet(adata)

### Normalization

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

### Feature Selection + Clustering

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
# Using the igraph implementation and a fixed number of iterations can be significantly faster,
# especially for larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=["leiden"])

# sc.pl.umap(adata, color=["leiden"], show).figure.savefig("output_path.ext")

In [ ]:
for res in [0.02, 0.5, 2.0]:
    sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

sc.pl.umap(
    adata,
    color=["leiden_res_0.02", "leiden_res_0.50", "leiden_res_2.00"],
    legend_loc="on data",
)

### Analysis of specific genes

In [ ]:
# Plot dotplot of marker genes for each cluster
marker_genes = {
    "Q1": ["Cd27"],
    "Q2": ["Cd27", "Esam"],
    "Q3": ["Esam"]
}

sc.pl.dotplot(adata, marker_genes, groupby="leiden_res_0.50", standard_scale="var")
sc.pl.stacked_violin(adata, marker_genes, groupby="leiden_res_0.50")

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0.50", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden_res_0.50", standard_scale="var", n_genes=5)

In [ ]:
# Get the table of differentially expressed genes
sc.get.rank_genes_groups_df(adata, group="7").head(5)
deg = sc.get.rank_genes_groups_df(adata, group=None)
deg.head()
deg.to_csv("C:/Users/rohit/OneDrive - Loyola University Chicago/Zhang Lab/RM MPC/differentially_expressed_genes.csv", index=False)

In [ ]:
# Get the names of the top 5 differentially expressed genes for cluster 7
dc_cluster_genes = sc.get.rank_genes_groups_df(adata, group="7").head(5)["names"]
sc.pl.umap(
    adata,
    color=[*dc_cluster_genes, "leiden_res_0.50"],
    legend_loc="on data",
    frameon=False,
    ncols=3,
)

## Trajectory Analysis

In [ ]:
sc.tl.paga(adata, groups="leiden_res_0.50")
sc.pl.paga(adata, color=["leiden_res_0.50", "Mpo", "Cd27", "Esam"])

In [ ]:
sc.tl.draw_graph(adata, init_pos="paga")
sc.pl.draw_graph(adata, color=["leiden_res_0.50", "Mpo", "Cd27", "Esam"], legend_loc="on data")